# 10-phase1-growing-decoder

Phase 1 (depth-only growing Transformer) 의 end-to-end 검증.

- **가설**:
  1. `PlateauTrigger` 가 학습 중 실제로 fire 한다
  2. `add_layer_function_preserving` 직후 loss 가 spike 없이 연속이다 (function preservation)
  3. 학습 종료 시 `n_layers > n_init_layers` 이면서 loss 가 감소한 상태다
- **데이터**: TinyShakespeare (char-LM, 첫 실행 시 자동 다운로드)
- **시드**: 42
- **작성일**: 2026-05-24
- **연관**: Issue [#23](https://github.com/EinSofINTEREST/GraphLM/issues/23) / PR [#24](https://github.com/EinSofINTEREST/GraphLM/pull/24)

> 노트북은 검증 흐름 + 시각화만 담당. 로직 (모델 / 학습 루프 / trigger) 은 모두 `src/graphlm/` 에서 import.

## 1. 환경 / 시드 / device

In [ ]:
import logging
import sys
from pathlib import Path

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.models.backbone import BackboneConfig, GrowingDecoder
from graphlm.training.loop import TrainConfig, train
from graphlm.utils.seed import set_seed

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(SEED)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("prefix  :", sys.prefix)
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)

## 2. 실험 설정

노트북용 smoke 검증 — `max_steps` 와 trigger 파라미터를 `scripts/train_tinyshakespeare.py` 기본값보다 공격적으로 (빨리 plateau 감지) 설정해서 1~2분 내에 grow event 가 1회 이상 발화하도록 함.

In [ ]:
DATA_PATH = "../../data/tinyshakespeare.txt"  # cache (gitignore)
OUT_DIR = Path("../../runs/notebook-phase1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16
BLOCK_SIZE = 128

BACKBONE = dict(
    hidden_dim=256,
    n_heads=4,
    ffn_dim=1024,
    n_init_layers=4,
)

TRAIN = dict(
    lr=3e-4,
    max_steps=1500,
    max_layers=8,
    trigger_window=100,
    trigger_epsilon=0.08,
    trigger_cooldown=150,
    trigger_min_history=100,  # PlateauTrigger 제약: min_history <= window (deque 가 window 에 capped)
    device=DEVICE,
)

for k, v in {**BACKBONE, **TRAIN}.items():
    print(f"  {k:22s} = {v}")

## 3. 데이터 로드 (char-LM)

첫 실행 시 `data/tinyshakespeare.txt` 가 없으면 `load_tinyshakespeare_text` 가 karpathy/char-rnn 에서 자동 다운로드 + 캐시.

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
data_iter = iter_random_batches(dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=SEED)

print(f"text length  : {len(text):>8d} chars")
print(f"vocab_size   : {tokenizer.vocab_size:>8d}")
print(
    f"batch shape  : {next(iter_random_batches(dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=SEED))[0].shape}"
)

## 4. 모델 초기화

`GrowingDecoder` 는 `n_init_layers=4` 의 `DecoderBlock` 으로 시작 (각 `α = 1.0`).

In [ ]:
cfg = BackboneConfig(
    vocab_size=tokenizer.vocab_size,
    max_seq_len=BLOCK_SIZE,
    **BACKBONE,
)
model = GrowingDecoder(cfg).to(DEVICE)

print(f"n_layers (init) : {model.n_layers}")
print(f"n_params (init) : {model.n_params:,}")
print(f"alpha vector    : {[b.alpha.item() for b in model.blocks]}")

## 5. 학습 (`train`)

`train(model, data_iter, cfg)` 가 내부적으로:
- `_train_step`: forward + CE + backward + AdamW.step
- `_maybe_grow`: 매 step 후 `trigger.update(loss)` → fire 시 `add_layer_function_preserving` + `optimizer.add_param_group`

CPU 에서 1500 step ≈ 수 분.

In [ ]:
train_cfg = TrainConfig(**TRAIN)
result = train(model, data_iter, train_cfg)

print("=" * 50)
print(f"final n_layers : {result.final_n_layers}")
print(f"final n_params : {result.final_n_params:,}")
print(f"grow events    : {len(result.grow_events)}")
for step, n in result.grow_events:
    print(f"  step={step:>5d}  ->  n_layers={n}")

n_init, n_last = min(100, len(result.losses)), min(100, len(result.losses))
first_avg = sum(result.losses[:n_init]) / n_init
last_avg = sum(result.losses[-n_last:]) / n_last
print(f"loss first {n_init} avg : {first_avg:.4f}")
print(f"loss last  {n_last} avg : {last_avg:.4f}")
print(f"improvement       : {first_avg - last_avg:+.4f}")

## 6. 시각화 — 손실 곡선 + grow event

빨간 수직선이 `add_layer_function_preserving` 발생 시점. 그 위치에서 loss 가 **spike 없이 연속** 이면 function preservation 이 학습 곡선에서도 확인된 것.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(result.losses) + 1), result.losses, lw=0.8, label="train loss")
for step, n in result.grow_events:
    ax.axvline(step, color="red", alpha=0.5, lw=1.0)
    ax.text(step, max(result.losses), f" L={n}", color="red", fontsize=8, va="top")
ax.set_xlabel("step")
ax.set_ylabel("CE loss")
ax.set_title("Phase 1 — GrowingDecoder on TinyShakespeare")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "curve.png", dpi=120)
plt.show()

## 7. Spike 정량 검증

각 grow event 직전/직후 N step 평균을 비교 — `|Δloss|` 가 임계 (예: 0.5) 이내면 spike 없음으로 판정.

In [ ]:
WINDOW = 30
SPIKE_THRESHOLD = 0.5

losses = result.losses
print(f"{'event step':>10}  {'L':>3}  {'pre avg':>8}  {'post avg':>8}  {'delta':>9}  spike?")
print("-" * 55)
all_ok = True
for step, n in result.grow_events:
    pre = losses[max(0, step - WINDOW) : step]
    post = losses[step : step + WINDOW]
    if not pre or not post:
        continue
    pre_avg = sum(pre) / len(pre)
    post_avg = sum(post) / len(post)
    delta = post_avg - pre_avg
    spike = abs(delta) > SPIKE_THRESHOLD
    all_ok &= not spike
    print(
        f"{step:>10}  {n:>3}  {pre_avg:>8.4f}  {post_avg:>8.4f}  {delta:>+9.4f}  {'YES' if spike else 'no'}"
    )
print("=" * 55)
print("function preservation:", "OK" if all_ok else "FAIL (spike detected)")

## 8. 학습된 α 분포

초기 block 의 α 는 1.0 부근에서 학습되었을 것이고, 후기에 추가된 block 의 α 는 0.0 에서 시작해 점차 양수 값으로 학습되었을 것.

In [ ]:
alphas = [b.alpha.item() for b in model.blocks]
print("final alpha per block:")
for i, a in enumerate(alphas):
    bar = "#" * max(1, int(abs(a) * 20))
    print(f"  block {i}: alpha = {a:+.4f}  {bar}")

## 9. 산출물 저장 (CSV)

In [ ]:
with (OUT_DIR / "losses.csv").open("w") as f:
    f.write("step,loss\n")
    for i, loss in enumerate(result.losses, 1):
        f.write(f"{i},{loss:.6f}\n")

with (OUT_DIR / "grow_events.csv").open("w") as f:
    f.write("step,n_layers_after\n")
    for step, n in result.grow_events:
        f.write(f"{step},{n}\n")

print("saved:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p}")

## 결과 요약 / 다음 가설

이 노트북에서 검증된 것:
- (1) PlateauTrigger 실제 fire 여부 → `len(result.grow_events) >= 1`
- (2) Function preservation 학습 곡선상 검증 → §7 의 `function preservation: OK`
- (3) 깊어진 모델 + 낮아진 loss → `final n_layers > n_init_layers` 이면서 `last_avg < first_avg`

**다음 가설 후보**:
- α 가 0 → 어떤 값으로 수렴하는 typical trajectory 가 있는가? (block 별)
- trigger 의 ε / window 가 grow event 수에 어떻게 영향?
- Phase 2 (FFN dim growth via LiGO) 로 가면 같은 step 예산에서 더 낮은 loss 도달 가능한가?